# 02 · Visión de Thomas — guantes y manos

**V2:** corrige las manchas negras de v1 con modo seguro CPU por defecto y controles de luminancia para la IA. No integrar una candidata sin inspección.

**Objetivo:** eliminar cualquier aspecto artificial de cuero/manos **sin modificar el bisturí**, su hoja, los adoquines, la sangre narrativa ni la disposición POV. Archivo original `web/assets/closeups/hands.jpg`, 1280×720.

El inpainting SOLO puede alterar las regiones dibujadas sobre los guantes; la fuente preserva su fondo pixel por pixel. Se procesan las dos manos en recortes independientes de 512×512 para la GPU T4. El original debe seguir reconocible: si aparecen dedos extra, falta de agarre, desaparece la hoja o cambia la iluminación de forma brusca, **rechaza el candidato**.

**Modelo de pesos abiertos, sin APIs de pago**; Colab GPU sí puede consumir unidades de Google AI Pro. No integra ni sube nada al repositorio.

In [ ]:
%pip -q install 'diffusers>=0.35,<0.40' 'transformers>=4.48,<5' 'accelerate>=1.0' 'safetensors>=0.5' 'Pillow>=10' 'numpy>=1.24' 'requests>=2.31'

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps, ImageChops
from IPython.display import display
import base64, io, json, math, hashlib, random, requests, shutil
REPO = "gabsvm/HorrorPlay"
COMMIT = "0681cfee36921ec2bd372f8ae8080a401f8bd63b"

def source(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": COMMIT}, headers={"Accept":"application/vnd.github+json"}, timeout=90)
    r.raise_for_status()
    data = r.json()
    if data.get("encoding") != "base64":
        raise RuntimeError(f"GitHub no devolvió el JPG/PNG en base64: {path}")
    b = base64.b64decode(data["content"])
    result = Image.open(io.BytesIO(b)).convert("RGB" if path.lower().endswith((".jpg", ".jpeg")) else "RGBA")
    return result, hashlib.sha256(b).hexdigest()

def font(size=20, bold=False):
    files = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
             "/usr/share/fonts/truetype/liberation2/LiberationSerif-Regular.ttf"]
    for f in files:
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def compare(images, thumb=(640,360), labels=None):
    w,h=thumb
    sheet=Image.new('RGB',(w*len(images),h+36),(17,18,20))
    dr=ImageDraw.Draw(sheet)
    for i,im in enumerate(images):
        sheet.paste(im.convert('RGB').resize((w,h), Image.Resampling.LANCZOS),(i*w,36))
        dr.text((i*w+12,10), labels[i] if labels else str(i),fill=(245,241,229),font=font(15))
    return sheet

def finish(folder, manifest, outputs):
    manifest.update({"repo":REPO,"source_commit":COMMIT,"status":"CANDIDATO - No integrado ni testeado en Android"})
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    archive=shutil.make_archive(str(folder),'zip',folder)
    print('Exportado:', archive, 'archivos:', len(list(folder.iterdir())))
    from google.colab import files
    files.download(archive)

In [ ]:
out=Path('/content/horrorplay_hands_candidates');out.mkdir(parents=True,exist_ok=True)
original, sha = source('web/assets/closeups/hands.jpg')
assert original.size==(1280,720)
print('Fuente sha256:',sha)
display(original.resize((960,540)))

REGIONS={
  # ROI(x1,y1,x2,y2), polygon en coordenadas del lienzo ORIGINAL
  'left':{
    'crop':(100,70,612,582),
    'polygon':[(234,405),(224,330),(259,254),(328,190),(409,178),(462,217),
               (487,260),(516,297),(483,335),(496,376),(442,426),(323,482)]},
  'right':{
    'crop':(644,55,1156,567),
    'polygon':[(724,270),(749,217),(777,167),(819,151),(865,176),(892,154),
               (928,166),(951,192),(981,193),(1010,226),(995,295),(1002,373),
               (931,451),(833,425),(780,355)]}
}
mask_full=Image.new('L',original.size,0);dr=ImageDraw.Draw(mask_full)
for reg in REGIONS.values():dr.polygon(reg['polygon'],fill=255)
# Proteger el bisturí, especialmente hoja y borde del mango en la mano izquierda.
dr.polygon([(520,49),(568,31),(527,156),(477,247),(445,238),(493,148)],fill=0)
red=Image.new('RGBA',original.size,(220,30,35,0));red.putalpha(mask_full.point(lambda n:round(n*.45)))
display(Image.alpha_composite(original.convert('RGBA'),red).resize((960,540)))
print('Verifica que rojo=guantes; el bisturí y fondo deben estar fuera.')

## Generación T4 (primero una seed)

Si la máscara no te convence, modifica `REGIONS` antes de ejecutar. El modelo puede reinterpretar el guante y fallar en dedos. Mantener `strength` bajo reduce cambios anatómicos. La generación no está garantizada y necesita revisión manual.

In [ ]:
# No descargar modelos ni consumir unidades de Colab salvo decisión explícita.
RUN_AI=False  # Cambiar a True únicamente si quieres comparar una seed de IA.
if RUN_AI:
    import torch
    from diffusers import AutoPipelineForInpainting
    assert torch.cuda.is_available(), 'Activa GPU T4 antes de usar RUN_AI'
    MODEL_ID='stable-diffusion-v1-5/stable-diffusion-inpainting'
    pipe=AutoPipelineForInpainting.from_pretrained(MODEL_ID,torch_dtype=torch.float16,
                         use_safetensors=False,variant='fp16')
    pipe.enable_model_cpu_offload();pipe.enable_vae_slicing()
    print('GPU y modelo:',torch.cuda.get_device_name(0),MODEL_ID)
else:
    print('Modo conservador CPU: mejora cuero original sin regenerar geometría. '
          'La generación IA es opcional y requiere activar RUN_AI=True.')


In [ ]:
import numpy as np
from PIL import ImageEnhance, ImageFilter, ImageStat
SEEDS=[1888]
PROMPT=('Victorian worn dark brown leather gloves, photorealistic material texture, original pose, '
        'two gloved hands, correct anatomy, unchanged grip and period clothing')
NEG=('black blob, missing glove, black void, amputated hand, missing fingers, extra fingers, '
     'duplicated arms, modern clothing, latex, plastic, new objects, new blade, cartoon, CGI')

# Corrección no generativa: realza ligeramente las costuras ya existentes.
# Máscara es sólo guante; excluir cualquier silueta, bisturí y adoquines.
feather=mask_full.filter(ImageFilter.GaussianBlur(4))
# Evitar que blur invada objetos protegidos.
feather=ImageChops.multiply(feather,mask_full)
clarified=ImageEnhance.Contrast(original).enhance(1.075)
clarified=clarified.filter(ImageFilter.UnsharpMask(radius=1.2,percent=38,threshold=4))
safe=Image.composite(clarified,original,feather)
assert safe.size==original.size
# Asegurar fuera de máscara la identidad píxel a píxel.
outside=Image.eval(mask_full,lambda v:255-v)
assert not ImageChops.multiply(ImageChops.difference(safe,original).convert('L'),outside).getbbox()
safe.save(out/'hands_conservative_1888.png')
safe.save(out/'hands_conservative_1888.jpg',quality=90,optimize=True)
compare([original,safe],(640,360),['Original','Conservador (sin IA)']).save(out/'AB_hands_safe_640.png')
display(compare([original,safe],(640,360),['Original','Conservador']))

accepted=[]
if RUN_AI:
    for seed in SEEDS:
        composite=original.copy()
        qa=[]
        for side,reg in REGIONS.items():
            crop=reg['crop'];input_crop=original.crop(crop)
            local_mask=mask_full.crop(crop)
            output=pipe(prompt=PROMPT,negative_prompt=NEG,image=input_crop,mask_image=local_mask,
              width=512,height=512,num_inference_steps=28,guidance_scale=6.0,strength=.23,
              generator=torch.Generator(device='cpu').manual_seed(seed+(0 if side=='left' else 1))).images[0].convert('RGB')
            oldarr=np.asarray(input_crop,dtype=np.int16)
            aiarr=np.asarray(output,dtype=np.int16)
            m=np.asarray(local_mask)>0
            oldlum=oldarr.mean(axis=2);newlum=aiarr.mean(axis=2)
            # Rechazar zonas negras que sustituyen el guante visible, defecto de v1.
            destroyed=((oldlum>52)&(newlum<oldlum*.42)&m)
            ratio=float(destroyed.sum()/max(1,m.sum()))
            print('Control',side,'seed',seed,'destrucción de luminancia',round(ratio,4))
            if ratio>.025:raise RuntimeError(f'RECHAZADO: mancha negra en {side}: {ratio:.1%}')
            # Guardián: cambios máximos <= 28/255 por canal para preservar cuero y pose.
            constrained=np.clip(aiarr,np.maximum(oldarr-28,0),np.minimum(oldarr+28,255)).astype(np.uint8)
            guarded=Image.blend(input_crop,Image.fromarray(constrained,'RGB'),.32)
            inner=ImageChops.multiply(local_mask.filter(ImageFilter.GaussianBlur(3)),local_mask)
            merged=Image.composite(guarded,input_crop,inner)
            composite.paste(merged,crop)
            qa.append({'side':side,'destroyed_fraction':ratio})
        assert not ImageChops.multiply(ImageChops.difference(composite,original).convert('L'),outside).getbbox()
        composite.save(out/f'hands_guarded_seed{seed}.png')
        accepted.append(composite)
    display(compare([original,*accepted],(640,360),['Original']+[f'IA controlada {s}' for s in SEEDS]))
    compare([original,*accepted],(640,360),['Original']+[f'IA {s}' for s in SEEDS]).save(out/'AB_hands_AI_640.png')
finish(out,{'source_sha256':sha,'ai_enabled':RUN_AI,'seed':SEEDS if RUN_AI else [],
   'fixed':'V2: black-blob rejector, fixed alpha mask, strict clamp, safe CPU baseline'},[])
